In [25]:
import duckdb

DB_PATH = "data/market_data.duckdb"
con = duckdb.connect(DB_PATH)


In [2]:
con.execute("""
CREATE OR REPLACE VIEW income_statement_calculated AS
SELECT
    fi.*,

    -- EBIT (calculated)
    CASE
        WHEN sm.industry = 'Financials' THEN fi.operating_profit - fi.depreciation
        ELSE fi.ebitda - fi.depreciation
    END AS ebit,


    -- Shares Outstanding (FIXED UNITS)
    CASE
        WHEN fi.eps IS NOT NULL AND fi.eps != 0
        THEN (COALESCE(fi.net_income_adj, 0) * 1e7) / fi.eps
        ELSE NULL
    END AS shares_outstanding

FROM financial_income_statement fi
JOIN stocks_master sm USING (stock_id)
""")

print("✅ income_statement_calculated created")


✅ income_statement_calculated created


In [3]:
con.execute("""
SELECT
    sm.symbol,
    period_end,
    ROUND(shares_outstanding / 1e7, 2) AS shares_outstanding_crore
FROM income_statement_calculated i
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol = 'RELIANCE'
ORDER BY period_end DESC
LIMIT 3;
""").df()


,symbol,period_end,shares_outstanding_crore
0,RELIANCE,2025-03-01,1353.18
1,RELIANCE,2024-03-01,1353.18
2,RELIANCE,2023-03-01,1353.26


In [4]:
con.execute("""
CREATE OR REPLACE VIEW quarterly_results_calculated AS
SELECT
    fq.*,

    -- EBIT (calculated)
    CASE
        WHEN sm.industry = 'Financials' THEN fq.operating_profit - fq.depreciation
        ELSE fq.ebitda - fq.depreciation
    END AS ebit,



    -- Shares Outstanding (derived from EPS)
    CASE
        WHEN fq.eps IS NOT NULL AND fq.eps != 0
        THEN COALESCE(fq.net_income_adj, 0) / fq.eps
        ELSE NULL
    END AS shares_outstanding

FROM financial_quarterly_results fq
JOIN stocks_master sm USING (stock_id)
""")

print("✅ quarterly_results_calculated created")


✅ quarterly_results_calculated created


In [5]:
con.execute("""
CREATE OR REPLACE VIEW balance_sheet_calculated AS
SELECT
    fb.*,

    -- Total Equity
    COALESCE(equity_share_capital, 0)
  + COALESCE(equity_reserves, 0)
    AS total_equity,

    -- Total Liabilities
    COALESCE(total_assets, 0)
  - (COALESCE(equity_share_capital, 0) + COALESCE(equity_reserves, 0))
    AS total_liabilities,

    -- Net Fixed Assets
    (
    COALESCE(fb.fixed_assets, 0)
  - COALESCE(fb.accumulated_depreciation, 0)
    )
    AS net_fixed_assets,

    -- Total Debt
    COALESCE(fb.borrowings, 0)

    AS total_debt,

    -- Net Debt
    (
    COALESCE(fb.borrowings, 0)

    )
  - COALESCE(fb.cash_and_equivalents, 0)
    AS net_debt,

    -- Invested Capital
    (
    COALESCE(fb.equity_share_capital, 0)
  + COALESCE(fb.equity_reserves, 0)
    )
  + (
    COALESCE(fb.borrowings, 0)

  - COALESCE(fb.cash_and_equivalents, 0)
    )
    AS invested_capital,


    -- Net Borrowing
    (
        COALESCE(borrowings, 0)
   
    )
  -
    LAG(
        COALESCE(borrowings, 0)
   
    ) OVER (
        PARTITION BY stock_id
        ORDER BY period_end
    )
    AS net_borrowing_bs










FROM financial_balance_sheet fb
""")

print("✅ balance_sheet_calculated created")








✅ balance_sheet_calculated created


In [6]:
con.execute("""
CREATE OR REPLACE VIEW shareholding_pattern_calculated AS
SELECT
    sp.*,

    -- Institutional Holdings = FIIs + DIIs
    COALESCE(sp.fii_holding, 0)
  + COALESCE(sp.dii_holding, 0)
    AS institutional_holding

FROM shareholding_pattern sp
""")

print("✅ shareholding_pattern_calculated created")


✅ shareholding_pattern_calculated created


In [7]:
con.execute("""
CREATE OR REPLACE VIEW cashflow_calculated AS
SELECT
    fc.*,

    -- CAPEX
    COALESCE(fc.fixed_assets_purchased, 0)
  + COALESCE(fc.fixed_assets_sold, 0)
    AS capex

    

FROM financial_cashflow fc
""")

print("✅ cashflow_calculated created")

✅ cashflow_calculated created


In [29]:
con.execute("""
SELECT
    sm.symbol,
    *

FROM  income_statement_calculated
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol = 'RELIANCE'
ORDER BY period_end DESC
LIMIT 3;
""").df()

,symbol,stock_id,period_end,revenue,ebitda,operating_profit,interest_expense,depreciation,ebt,tax_expense_percent,net_income,net_income_adj,eps,dividend_Payout_ratio,ebit,shares_outstanding,symbol_1,company_name,sector,industry
0,RELIANCE,3,2025-03-01,962820.0,165598.0,NaN,24269.0,53136.0,106017.0,24.0,81309.0,69648.0,51.47,11.0,112462.0,1.353177e+10,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
1,RELIANCE,3,2024-03-01,899041.0,162498.0,NaN,23118.0,50832.0,104340.0,25.0,79020.0,69621.0,51.45,10.0,111666.0,1.353178e+10,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
2,RELIANCE,3,2023-03-01,876396.0,142318.0,NaN,19571.0,40303.0,94464.0,22.0,74088.0,66702.0,49.29,9.0,102015.0,1.353256e+10,RELIANCE,Reliance Industries Ltd,Energy,Non Financials


In [28]:
con.execute("""
SELECT
    sm.symbol,
    *

FROM cashflow_calculated 
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol = 'RELIANCE'
ORDER BY period_end DESC
LIMIT 3;
""").df()


,symbol,stock_id,period_end,profit_from_operations,receivables,inventory,payables,direct_taxes,loans_advances,operating_investments,...,financial_liabilities,share_application_money,other_financing_items,cash_from_financing_activity,net_cash_flow,capex,symbol_1,company_name,sector,industry
0,RELIANCE,3,2025-03-01,166904.0,-17837.0,3134.0,38427.0,-11925.0,NaN,NaN,...,-2956.0,0.0,22.0,-31891.0,9277.0,-137624.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
1,RELIANCE,3,2024-03-01,164383.0,-15674.0,-12756.0,34796.0,-11961.0,NaN,NaN,...,-2483.0,0.0,-1078.0,-16646.0,28561.0,-137576.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
2,RELIANCE,3,2023-03-01,140963.0,13194.0,-32228.0,-600.0,-6297.0,NaN,NaN,...,-1406.0,0.0,40.0,10455.0,32486.0,-131802.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials


In [8]:
con.execute("""
CREATE OR REPLACE VIEW income_statement_annual_growth AS
SELECT
    isc.stock_id,
    isc.period_end,

    isc.revenue,
    isc.net_income_adj,
    isc.eps,

    -- Revenue YoY Growth
    (isc.revenue
     - LAG(isc.revenue) OVER (PARTITION BY isc.stock_id ORDER BY isc.period_end))
    / NULLIF(LAG(isc.revenue) OVER (PARTITION BY isc.stock_id ORDER BY isc.period_end), 0)
        AS revenue_yoy_growth,

    -- Net Income YoY Growth
    (isc.net_income
     - LAG(isc.net_income_adj) OVER (PARTITION BY isc.stock_id ORDER BY isc.period_end))
    / NULLIF(LAG(isc.net_income_adj) OVER (PARTITION BY isc.stock_id ORDER BY isc.period_end), 0)
        AS profit_yoy_growth,

    -- EPS YoY Growth
    (isc.eps
     - LAG(isc.eps) OVER (PARTITION BY isc.stock_id ORDER BY isc.period_end))
    / NULLIF(LAG(isc.eps) OVER (PARTITION BY isc.stock_id ORDER BY isc.period_end), 0)
        AS eps_yoy_growth

FROM income_statement_calculated isc
""")

print("✅ income_statement_annual_growth created")


✅ income_statement_annual_growth created


In [9]:
#ANNUAL YoY GROWTH VIEW

In [10]:
con.execute("""
CREATE OR REPLACE VIEW income_statement_quarterly_qoq_growth AS
SELECT
    fq.stock_id,
    fq.period_end,

    fq.revenue,
    fq.net_income_adj,
    fq.eps,

    -- Revenue QoQ Growth
    (fq.revenue
     - LAG(fq.revenue) OVER (PARTITION BY fq.stock_id ORDER BY fq.period_end))
    / NULLIF(LAG(fq.revenue) OVER (PARTITION BY fq.stock_id ORDER BY fq.period_end), 0)
        AS revenue_qoq_growth,

    -- Net Income QoQ Growth
    (fq.net_income
     - LAG(fq.net_income_adj) OVER (PARTITION BY fq.stock_id ORDER BY fq.period_end))
    / NULLIF(LAG(fq.net_income_adj) OVER (PARTITION BY fq.stock_id ORDER BY fq.period_end), 0)
        AS profit_qoq_growth,

    -- EPS QoQ Growth
    (fq.eps
     - LAG(fq.eps) OVER (PARTITION BY fq.stock_id ORDER BY fq.period_end))
    / NULLIF(LAG(fq.eps) OVER (PARTITION BY fq.stock_id ORDER BY fq.period_end), 0)
        AS eps_qoq_growth

FROM financial_quarterly_results fq
""")

print("✅ income_statement_quarterly_qoq_growth created")


✅ income_statement_quarterly_qoq_growth created


In [11]:
#QUARTERLY QoQ GROWTH

In [12]:
#QUARTERLY YoY GROWTH (Seasonality-safe)

In [13]:
con.execute("""
CREATE OR REPLACE VIEW income_statement_quarterly_yoy_growth AS
SELECT
    fq.stock_id,
    fq.period_end,

    fq.revenue,
    fq.net_income_adj,
    fq.eps,

    -- Revenue YoY Growth (Quarterly)
    (fq.revenue
     - LAG(fq.revenue, 4) OVER (PARTITION BY fq.stock_id ORDER BY fq.period_end))
    / NULLIF(LAG(fq.revenue, 4) OVER (PARTITION BY fq.stock_id ORDER BY fq.period_end), 0)
        AS revenue_yoy_growth,

    -- Net Income YoY Growth (Quarterly)
    (fq.net_income
     - LAG(fq.net_income_adj, 4) OVER (PARTITION BY fq.stock_id ORDER BY fq.period_end))
    / NULLIF(LAG(fq.net_income_adj, 4) OVER (PARTITION BY fq.stock_id ORDER BY fq.period_end), 0)
        AS profit_yoy_growth,

    -- EPS YoY Growth (Quarterly)
    (fq.eps
     - LAG(fq.eps, 4) OVER (PARTITION BY fq.stock_id ORDER BY fq.period_end))
    / NULLIF(LAG(fq.eps, 4) OVER (PARTITION BY fq.stock_id ORDER BY fq.period_end), 0)
        AS eps_yoy_growth

FROM financial_quarterly_results fq
""")

print("✅ income_statement_quarterly_yoy_growth created")


✅ income_statement_quarterly_yoy_growth created


In [14]:
#✅ ANNUAL 3Y & 5Y CAGR VIEW

In [15]:
con.execute("""
CREATE OR REPLACE VIEW income_statement_annual_cagr AS
SELECT
    isc.stock_id,
    isc.period_end,

    isc.revenue,
    isc.net_income_adj,
    isc.eps,

    /* =========================
       3Y CAGR
       ========================= */

    CASE
        WHEN isc.revenue > 0
         AND LAG(isc.revenue, 3) OVER w > 0
        THEN
            POWER(
                isc.revenue
                / LAG(isc.revenue, 3) OVER w,
                1.0 / 3
            ) - 1
        ELSE NULL
    END AS revenue_cagr_3y,

    CASE
        WHEN isc.net_income > 0
         AND LAG(isc.net_income_adj, 3) OVER w > 0
        THEN
            POWER(
                isc.net_income
                / LAG(isc.net_income_adj, 3) OVER w,
                1.0 / 3
            ) - 1
        ELSE NULL
    END AS profit_cagr_3y,

    CASE
        WHEN isc.eps > 0
         AND LAG(isc.eps, 3) OVER w > 0
        THEN
            POWER(
                isc.eps
                / LAG(isc.eps, 3) OVER w,
                1.0 / 3
            ) - 1
        ELSE NULL
    END AS eps_cagr_3y,

    /* =========================
       5Y CAGR
       ========================= */

    CASE
        WHEN isc.revenue > 0
         AND LAG(isc.revenue, 5) OVER w > 0
        THEN
            POWER(
                isc.revenue
                / LAG(isc.revenue, 5) OVER w,
                1.0 / 5
            ) - 1
        ELSE NULL
    END AS revenue_cagr_5y,

    CASE
        WHEN isc.net_income > 0
         AND LAG(isc.net_income_adj, 5) OVER w > 0
        THEN
            POWER(
                isc.net_income
                / LAG(isc.net_income_adj, 5) OVER w,
                1.0 / 5
            ) - 1
        ELSE NULL
    END AS profit_cagr_5y,

    CASE
        WHEN isc.eps > 0
         AND LAG(isc.eps, 5) OVER w > 0
        THEN
            POWER(
                isc.eps
                / LAG(isc.eps, 5) OVER w,
                1.0 / 5
            ) - 1
        ELSE NULL
    END AS eps_cagr_5y

FROM income_statement_calculated isc

WINDOW w AS (
    PARTITION BY isc.stock_id
    ORDER BY isc.period_end
)
""")

print("✅ income_statement_annual_cagr created (3Y & 5Y)")


✅ income_statement_annual_cagr created (3Y & 5Y)


In [16]:
#✅ GROWTH CONSISTENCY SCORE — FINAL CODEx`

In [17]:
con.execute("""
CREATE OR REPLACE VIEW growth_consistency_score AS
WITH base AS (
    SELECT
        stock_id,
        revenue_yoy_growth
    FROM income_statement_annual_growth
    WHERE revenue_yoy_growth IS NOT NULL
),

aggregates AS (
    SELECT
        stock_id,

        COUNT(*) AS total_years,

        -- Part A: Positive growth years
        SUM(
            CASE
                WHEN revenue_yoy_growth > 0 THEN 1
                ELSE 0
            END
        ) AS positive_years,

        -- Part B: Volatility (std dev of growth)
        STDDEV_SAMP(revenue_yoy_growth) AS growth_volatility

    FROM base
    GROUP BY stock_id
)

SELECT
    a.stock_id,

    a.total_years,
    a.positive_years,

    -- Positive Year Ratio
    CASE
        WHEN a.total_years > 0
        THEN a.positive_years * 1.0 / a.total_years
        ELSE NULL
    END AS positive_year_ratio,

    -- Volatility Penalty
    a.growth_volatility,

    -- Final Growth Consistency Score
    CASE
        WHEN a.total_years > 0
        THEN
            (a.positive_years * 1.0 / a.total_years)
            * (1.0 / (1.0 + COALESCE(a.growth_volatility, 0)))
        ELSE NULL
    END AS growth_consistency_score

FROM aggregates a
""")

print("✅ growth_consistency_score created")


✅ growth_consistency_score created


In [18]:
#✅ FINANCIAL RATIOS — FINAL CALCULATED VIEW

In [19]:
con.execute("""
CREATE OR REPLACE VIEW financial_ratios_calculated AS
SELECT
    isc.stock_id,
    isc.period_end,

    /* =========================
       LEVERAGE RATIOS
       ========================= */

    -- Debt to Equity
    CASE
        WHEN bsc.total_equity != 0
        THEN bsc.total_debt / bsc.total_equity
        ELSE NULL
    END AS debt_to_equity,

    -- Debt Ratio
    CASE
        WHEN bsc.total_assets != 0
        THEN bsc.total_debt / bsc.total_assets
        ELSE NULL
    END AS debt_ratio,

    /* =========================
       COVERAGE RATIO
       ========================= */

    -- Interest Coverage Ratio
    CASE
        WHEN isc.interest_expense IS NOT NULL
         AND isc.interest_expense != 0
        THEN isc.ebit / isc.interest_expense
        ELSE NULL
    END AS interest_coverage_ratio,

    /* =========================
       MARGINS
       ========================= */

    -- EBITDA Margin
    CASE
        WHEN isc.revenue != 0
        THEN isc.ebitda / isc.revenue
        ELSE NULL
    END AS ebitda_margin,

    -- EBIT Margin
    CASE
        WHEN isc.revenue != 0
        THEN isc.ebit / isc.revenue
        ELSE NULL
    END AS ebit_margin,

    -- EBT Margin
    CASE
        WHEN isc.revenue != 0
        THEN isc.ebt / isc.revenue
        ELSE NULL
    END AS ebt_margin,

    -- Net Income Margin (Adjusted)
    CASE
        WHEN isc.revenue != 0
        THEN isc.net_income_adj / isc.revenue
        ELSE NULL
    END AS net_income_margin,

    /* =========================
       RETURN RATIOS
       ========================= */

    -- Return on Equity (ROE)
    CASE
        WHEN bsc.total_equity != 0
        THEN isc.net_income / bsc.total_equity
        ELSE NULL
    END AS return_on_equity,

    -- Return on Invested Capital (ROIC)
    CASE
        WHEN bsc.invested_capital != 0
        THEN
            (
                isc.ebit
                * (1 - COALESCE(isc.tax_expense_percent, 0) / 100.0)
            ) / bsc.invested_capital
        ELSE NULL
    END AS return_on_invested_capital,

    -- Return on Assets (ROA)
    CASE
        WHEN bsc.total_assets != 0
        THEN isc.net_income / bsc.total_assets
        ELSE NULL
    END AS return_on_assets,

    /* =========================
       EFFICIENCY RATIOS
       ========================= */

    -- Asset Turnover
    CASE
        WHEN bsc.total_assets != 0
        THEN isc.revenue / bsc.total_assets
        ELSE NULL
    END AS asset_turnover,

    -- Receivables Turnover
    CASE
        WHEN bsc.trade_receivables IS NOT NULL
         AND bsc.trade_receivables != 0
        THEN isc.revenue / bsc.trade_receivables
        ELSE NULL
    END AS receivables_turnover

FROM income_statement_calculated isc
JOIN balance_sheet_calculated bsc
  ON isc.stock_id = bsc.stock_id
 AND isc.period_end = bsc.period_end
JOIN stocks_master sm
  ON isc.stock_id = sm.stock_id
""")

print("✅ financial_ratios_calculated created")


✅ financial_ratios_calculated created


In [20]:
con.execute("""
CREATE OR REPLACE VIEW financial_ratios_3y_5y_avg AS
SELECT
    fr.stock_id,
    fr.period_end,

    /* =========================
       LEVERAGE RATIOS
       ========================= */

    AVG(fr.debt_to_equity) OVER w3 AS debt_to_equity_3y_avg,
    AVG(fr.debt_to_equity) OVER w5 AS debt_to_equity_5y_avg,

    AVG(fr.debt_ratio) OVER w3 AS debt_ratio_3y_avg,
    AVG(fr.debt_ratio) OVER w5 AS debt_ratio_5y_avg,

    /* =========================
       COVERAGE
       ========================= */

    AVG(fr.interest_coverage_ratio) OVER w3 AS interest_coverage_3y_avg,
    AVG(fr.interest_coverage_ratio) OVER w5 AS interest_coverage_5y_avg,

    /* =========================
       MARGINS
       ========================= */

    AVG(fr.ebitda_margin) OVER w3 AS ebitda_margin_3y_avg,
    AVG(fr.ebitda_margin) OVER w5 AS ebitda_margin_5y_avg,

    AVG(fr.ebit_margin) OVER w3 AS ebit_margin_3y_avg,
    AVG(fr.ebit_margin) OVER w5 AS ebit_margin_5y_avg,

    AVG(fr.ebt_margin) OVER w3 AS ebt_margin_3y_avg,
    AVG(fr.ebt_margin) OVER w5 AS ebt_margin_5y_avg,

    AVG(fr.net_income_margin) OVER w3 AS net_income_margin_3y_avg,
    AVG(fr.net_income_margin) OVER w5 AS net_income_margin_5y_avg,

    /* =========================
       RETURN RATIOS
       ========================= */

    AVG(fr.return_on_equity) OVER w3 AS roe_3y_avg,
    AVG(fr.return_on_equity) OVER w5 AS roe_5y_avg,

    AVG(fr.return_on_invested_capital) OVER w3 AS roic_3y_avg,
    AVG(fr.return_on_invested_capital) OVER w5 AS roic_5y_avg,

    AVG(fr.return_on_assets) OVER w3 AS roa_3y_avg,
    AVG(fr.return_on_assets) OVER w5 AS roa_5y_avg,

    /* =========================
       EFFICIENCY
       ========================= */

    AVG(fr.asset_turnover) OVER w3 AS asset_turnover_3y_avg,
    AVG(fr.asset_turnover) OVER w5 AS asset_turnover_5y_avg,

    AVG(fr.receivables_turnover) OVER w3 AS receivables_turnover_3y_avg,
    AVG(fr.receivables_turnover) OVER w5 AS receivables_turnover_5y_avg

FROM financial_ratios_calculated fr

WINDOW
    w3 AS (
        PARTITION BY fr.stock_id
        ORDER BY fr.period_end
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ),
    w5 AS (
        PARTITION BY fr.stock_id
        ORDER BY fr.period_end
        ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
    )
""")

print("✅ financial_ratios_3y_5y_avg created")


✅ financial_ratios_3y_5y_avg created


In [21]:
con.execute("""
SELECT
 *
FROM  financial_ratios_3y_5y_avg fi
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol = 'RELIANCE'

LIMIT 20
""").df()

,stock_id,period_end,debt_to_equity_3y_avg,debt_to_equity_5y_avg,debt_ratio_3y_avg,debt_ratio_5y_avg,interest_coverage_3y_avg,interest_coverage_5y_avg,ebitda_margin_3y_avg,ebitda_margin_5y_avg,...,roa_3y_avg,roa_5y_avg,asset_turnover_3y_avg,asset_turnover_5y_avg,receivables_turnover_3y_avg,receivables_turnover_5y_avg,symbol,company_name,sector,industry
0,3,2014-03-01,0.698390,0.698390,0.323571,0.323571,6.187174,6.187174,0.080584,0.080584,...,0.052579,0.052579,1.010908,1.010908,46.065349,46.065349,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
1,3,2015-03-01,0.734210,0.734210,0.328540,0.328540,6.999196,6.999196,0.090308,0.090308,...,0.049719,0.049719,0.876497,0.876497,58.251113,58.251113,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
2,3,2016-03-01,0.769772,0.769772,0.327382,0.327382,7.394931,7.394931,0.111298,0.111298,...,0.049763,0.049763,0.736020,0.736020,59.183683,59.183683,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
3,3,2017-03-01,0.811868,0.783498,0.322088,0.322459,8.334271,7.797497,0.135219,0.121561,...,0.046307,0.047875,0.542398,0.659525,56.219175,53.680718,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
4,3,2018-03-01,0.827579,0.790232,0.309464,0.317095,7.701429,7.420536,0.156730,0.130161,...,0.045511,0.047194,0.455616,0.623968,40.161141,47.397130,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
5,3,2019-03-01,0.812247,0.809533,0.303924,0.314070,6.252128,6.950801,0.155050,0.143692,...,0.042204,0.044665,0.493823,0.535724,26.107709,41.961766,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
6,3,2020-03-01,0.800904,0.813657,0.303146,0.308439,4.265257,5.997473,0.154136,0.153607,...,0.039565,0.042151,0.521491,0.489916,23.835793,33.945605,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
7,3,2021-03-01,0.661322,0.725162,0.275042,0.285690,3.147286,4.871949,0.157033,0.157602,...,0.038310,0.040323,0.478659,0.469552,24.589669,26.640721,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
8,3,2022-03-01,0.532839,0.642116,0.243216,0.266746,3.668825,4.151533,0.159722,0.158393,...,0.040090,0.040935,0.443277,0.476252,28.088656,25.083455,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
9,3,2023-03-01,0.479598,0.604868,0.235183,0.263870,4.391484,4.011506,0.163983,0.157959,...,0.044039,0.041268,0.454176,0.489052,28.238947,26.792289,RELIANCE,Reliance Industries Ltd,Energy,Non Financials


In [30]:
con.close()